In [ ]:
from libraries.inference_training import Configuration, ImageDataset
from libraries.inference_training import initCudaEnvironment, createTransforms
from libraries.inference_training import drawImageAndFeatureMasks
from libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from libraries.inference_training import trainModel, saveModel, loadModel
from libraries.inference_training import createModelInstance, testInference
from libraries.engine import evaluate
import libraries.utils as utils
import torch
import os
import random

In [ ]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

# load model

In [ ]:
def load_model(path:str, config=None):
    """
    :param path: path naar model save location 
    :param config: config als het eerder is ingesteld
    :return: 
    """
    if not config:
        config = Configuration()
    
    model = createModelInstance(config)
    loadModel(config, model, path)
    
    return model, config

In [ ]:
def load_default_config():
    config = Configuration()
    config.setIsCrowd(False)
    config.setFilePrefix("")
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.addLegendEntry("Background", 0, "#00000000")
    config.setOnnxMetaData(scoreThreshold=0.2,
                           maskThreshold=0.3,
                           strideFraction=0.5)
    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    
    return config

# average recall/average precision evaluation based on different IOU prerequisites template

In [ ]:
model_path = "<insert path to model here>"
eval_path = "<insert path to evaluation dataset here>"
config = load_default_config()
config.setDatasetPaths(testPath=eval_path)
evalDataset = ImageDataset(config, False, createTransforms(False))
config = load_default_config()
model, config = load_model(model_path,config)
testDataLoader = torch.utils.data.DataLoader(
    evalDataset,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn
)
evaluate(model, testDataLoader, device=config.device)

TODO: DICE score evaluation template, mAP score evaluation template

# 25 epoch combo model evaluation

In [ ]:
model_path = "C:/xxx/models/combo_models/combo_sets_model.pt"
trainPath = "C:/xxx/datasets/combo_overlay_sets/train"
eval_path = "C:/xxx/datasets/combo_overlay_sets/eval"
config = load_default_config()
config.setDatasetPaths(trainPath= trainPath, testPath=eval_path)
evalDataset = ImageDataset(config, False, createTransforms(False))
config = load_default_config()
model, config = load_model(model_path,config)
testDataLoader = torch.utils.data.DataLoader(
    evalDataset,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn
)
evaluate(model, testDataLoader, device=config.device)

# 25 epoch augmented combo model evaluation

In [ ]:
model_path = "C:/xxx/models/combo_models/augment/combo_model_with_augment_epoch_25.pt"
trainPath = "C:/xxx/datasets/combo_overlay_sets/train"
eval_path = "C:/xxx/datasets/combo_overlay_sets/eval"
config = load_default_config()
config.setDatasetPaths(trainPath=trainPath, testPath=eval_path)
evalDataset = ImageDataset(config, False, createTransforms(False))
config = load_default_config()
model, config = load_model(model_path,config)
testDataLoader = torch.utils.data.DataLoader(
    evalDataset,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn
)
evaluate(model, testDataLoader, device=config.device)